# prbd01n001 결측 최대 보완 파이프라인

이 노트북은 원본 파일을 수정하지 않고, 실제 NULL인 셀만 보완합니다. 기존의 0, 빈 문자열, 기존 비결측값은 보호됩니다. 모든 자동 보완은 출처, 기준일, 매칭 키, 신뢰도를 감사 로그에 남깁니다.

보완 순서는 다음과 같습니다.

1. 동일 ISIN 내부의 유일한 정적값
2. SEIBro ISIN 상세정보(키 불필요)
3. 금융위원회·한국예탁결제원 채권기본정보 API
4. 금융위원회·한국거래소 채권시세 API
5. 선택 파일: 신용등급, 채권평가, 내부 판매·재고 원장
6. 만기일이 확정된 경우 잔존일수 계산

주의: 판매수익률·매매단가·매수가능수량·판매채널 같은 13개 판매 블록은 공개 시장정보가 아니라 회사 내부 판매/재고 스냅샷입니다. 공개 API 값으로 대신 채우지 않으며, 정확한 내부 원장을 연결했을 때만 채웁니다. 장외채권의 장내 종가, 무등급 국공채의 신용등급처럼 개념상 해당하지 않는 결측도 값으로 만들지 않습니다.

공식 API 안내: [채권기본정보](https://www.data.go.kr/data/15059592/openapi.do), [채권시세정보](https://www.data.go.kr/data/15094784/openapi.do), [OpenDART](https://opendart.fss.or.kr/guide/main.do)

## 0. 실행 설정

공공데이터포털의 두 인증키를 각각 BOND_BASIC_SERVICE_KEY와 KRX_BOND_PRICE_SERVICE_KEY에 넣습니다. 환경변수를 사용하지 않는 경우 아래 코드 셀의 빈 문자열에 직접 붙여 넣어도 됩니다. 키가 없어도 동일 ISIN 보완과 SEIBro 단계는 실행됩니다. 외부 파일 경로는 비워두면 자동으로 건너뜁니다. API 호출은 중간 결과를 캐시에 저장하므로 중단 후 다시 실행해도 이어서 처리할 수 있습니다.

In [ ]:
from pathlib import Path
from io import StringIO, BytesIO
from urllib.parse import urlencode, quote_plus
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import re
import time
import warnings
import xml.etree.ElementTree as ET
import zipfile

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

# 환경변수를 쓰지 않는 경우 두 번째 인수인 빈 문자열에 각 Decoding 인증키를 붙여 넣으세요.
BOND_BASIC_SERVICE_KEY = ""
KRX_BOND_PRICE_SERVICE_KEY = ""
OPENDART_API_KEY = os.getenv('OPENDART_API_KEY', '').strip()

RUN_SEIBRO = os.getenv('RUN_SEIBRO', '1') == '1'
RUN_PUBLIC_DATA_API = os.getenv('RUN_PUBLIC_DATA_API', '1') == '1'
RUN_KRX_PRICE_API = os.getenv('RUN_KRX_PRICE_API', '1') == '1'
RUN_DART_CANDIDATES = os.getenv('RUN_DART_CANDIDATES', '0') == '1'
WRITE_OUTPUTS = os.getenv('WRITE_OUTPUTS', '1') == '1'

SEIBRO_MAX_ISINS = int(os.getenv('SEIBRO_MAX_ISINS', '1000'))
SEIBRO_DELAY_SECONDS = float(os.getenv('SEIBRO_DELAY_SECONDS', '0.40'))
PUBLIC_API_PAGE_SIZE = int(os.getenv('PUBLIC_API_PAGE_SIZE', '1000'))
PUBLIC_API_MAX_ISINS = int(os.getenv('PUBLIC_API_MAX_ISINS', '1000'))

RATINGS_FILE = os.getenv('RATINGS_FILE', '').strip()
PRICING_FILE = os.getenv('PRICING_FILE', '').strip()
INTERNAL_SALES_FILE = os.getenv('INTERNAL_SALES_FILE', '').strip()

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'prbd01n001_external_fill_output'
CACHE_DIR = OUTPUT_DIR / 'cache'
SEIBRO_CACHE_DIR = CACHE_DIR / 'seibro'
if WRITE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    SEIBRO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def discover_data_file():
    explicit = os.getenv('PRBD01N001_DATA_FILE', '').strip()
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    candidates.extend([ROOT / 'prbd01n001_data.xlsx', ROOT.parent / 'prbd01n001_data.xlsx'])
    metadata_paths = [ROOT / 'prbd01n001_missingness_output' / 'analysis_metadata.json', ROOT / 'analysis_metadata.json']
    for meta_path in metadata_paths:
        if meta_path.exists():
            try:
                candidates.append(Path(json.loads(meta_path.read_text(encoding='utf-8'))['data_file']))
            except Exception:
                pass
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError('prbd01n001_data.xlsx를 찾지 못했습니다. PRBD01N001_DATA_FILE 환경변수에 전체 경로를 지정하세요.')

DATA_FILE = discover_data_file()
print('원본:', DATA_FILE)
print('채권기본정보 키:', '설정됨' if BOND_BASIC_SERVICE_KEY else '없음 - 해당 단계 자동 생략')
print('KRX 채권시세 키:', '설정됨' if KRX_BOND_PRICE_SERVICE_KEY else '없음 - 해당 단계 자동 생략')
print('SEIBro:', '실행' if RUN_SEIBRO else '생략')
print('출력 폴더:', OUTPUT_DIR)

## 1. 원본 로드와 보호 장치

결측 여부는 pd.isna()만 사용합니다. 따라서 0과 빈 문자열은 결측으로 바꾸지 않습니다. 원본의 행 순서와 열 순서도 유지합니다.

In [ ]:
df_original = pd.read_excel(DATA_FILE)
df_original = df_original.reset_index(drop=True)
df = df_original.copy(deep=True)
original_columns = df_original.columns.tolist()
original_nonnull = df_original.notna()
original_null = df_original.isna()

def normalize_date(value):
    if pd.isna(value):
        return None
    parsed = pd.to_datetime(value, errors='coerce')
    if pd.isna(parsed):
        digits = re.sub(r'[^0-9]', '', str(value))
        parsed = pd.to_datetime(digits, format='%Y%m%d', errors='coerce')
    return None if pd.isna(parsed) else parsed.normalize()

def date8(value):
    parsed = normalize_date(value)
    return '' if parsed is None else parsed.strftime('%Y%m%d')

def normalize_text(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

def candidate_available(value):
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except Exception:
        pass
    return not (isinstance(value, str) and value.strip() == '')

def row_key(index):
    pd_no = normalize_text(df_original.at[index, 'pd_no']) if 'pd_no' in df_original else ''
    base_dt = date8(df_original.at[index, 'info_base_dt']) if 'info_base_dt' in df_original else ''
    market = normalize_text(df_original.at[index, 'pd_exg_mkt']) if 'pd_exg_mkt' in df_original else ''
    seq = normalize_text(df_original.at[index, 'info_seq']) if 'info_seq' in df_original else ''
    raw = f'{index}|{pd_no}|{base_dt}|{market}|{seq}'
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:20]

ROW_KEYS = pd.Series([row_key(i) for i in df.index], index=df.index, name='row_key')
audit_records = []
review_records = []

def record_review(index, column, candidate, source, source_date='', match_key='', confidence='REVIEW', reason=''):
    review_records.append({
        'row_index': int(index), 'row_key': ROW_KEYS.at[index],
        'pd_no': df_original.at[index, 'pd_no'] if 'pd_no' in df_original else None,
        'column': column, 'candidate': candidate, 'source': source,
        'source_date': source_date, 'match_key': match_key,
        'confidence': confidence, 'reason': reason
    })

def safe_fill(index, column, value, source, source_date='', match_key='', confidence='EXACT', notes='', apply=True):
    if column not in df.columns or index not in df.index or not candidate_available(value):
        return False
    if pd.notna(df_original.at[index, column]):
        return False
    if pd.notna(df.at[index, column]):
        return False
    if not apply:
        record_review(index, column, value, source, source_date, match_key, confidence, notes or '자동 적용 비활성화')
        return False
    df.at[index, column] = value
    audit_records.append({
        'row_index': int(index), 'row_key': ROW_KEYS.at[index],
        'pd_no': df_original.at[index, 'pd_no'] if 'pd_no' in df_original else None,
        'info_base_dt': df_original.at[index, 'info_base_dt'] if 'info_base_dt' in df_original else None,
        'column': column, 'filled_value': value, 'source': source,
        'source_date': source_date, 'match_key': match_key,
        'confidence': confidence, 'notes': notes
    })
    return True

raw_null_counts = df_original.isna().sum().sort_values(ascending=False)
print(f'행 {len(df_original):,} / 열 {len(df_original.columns):,}')
print(f'실제 NULL 셀 {int(df_original.isna().sum().sum()):,} / NULL 열 {(raw_null_counts > 0).sum()}개')
display(raw_null_counts[raw_null_counts > 0].rename('raw_null_count').to_frame())

## 2. 동일 ISIN 내부의 유일한 정적값

같은 pd_no 안에서 비결측값이 정확히 하나뿐인 정적 변수만 복사합니다. 여러 값이 관찰되면 아무것도 채우지 않습니다. 판매수익률, 가격, 잔액, 평가값처럼 시점이나 LOT에 따라 변할 수 있는 변수는 이 단계에서 제외합니다.

In [ ]:
STATIC_SAME_ISIN_COLUMNS = [
    'pd_nm', 'pd_eng_nm', 'pd_abrv_nm', 'pd_abrv_eng_nm', 'pd_pbcm',
    'pd_ctry_cd', 'curr_cd', 'bd_knd', 'bd_inrt_tcd', 'bd_intp_tcd',
    'bd_ofr_tcd', 'isu_dt', 'mat_dt', 'srfc_irt',
    'std_pd_mcls_nm', 'std_pd_scls_nm', 'crd_grd', 'crd_grd_dt'
]

def canonical_unique(values):
    cleaned = [v for v in values if candidate_available(v)]
    if not cleaned:
        return None
    normalized = {}
    for value in cleaned:
        if isinstance(value, (pd.Timestamp, datetime)):
            key = 'date:' + date8(value)
        elif isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
            key = 'number:' + format(float(value), '.12g')
        else:
            key = 'text:' + normalize_text(value)
        normalized.setdefault(key, value)
    return next(iter(normalized.values())) if len(normalized) == 1 else None

before_internal = len(audit_records)
if 'pd_no' in df.columns:
    for column in [c for c in STATIC_SAME_ISIN_COLUMNS if c in df.columns]:
        unique_map = df_original.groupby('pd_no', dropna=False)[column].apply(canonical_unique)
        for idx in df.index[df[column].isna()]:
            isin = df_original.at[idx, 'pd_no']
            if pd.isna(isin) or isin not in unique_map.index:
                continue
            value = unique_map.loc[isin]
            safe_fill(idx, column, value, 'internal_same_isin_unique',
                      source_date=date8(df_original.at[idx, 'info_base_dt']) if 'info_base_dt' in df_original else '',
                      match_key=f'pd_no={isin}', confidence='EXACT_INTERNAL',
                      notes='동일 ISIN에서 비결측값이 정확히 하나인 정적 변수')
print('동일 ISIN 내부 보완:', len(audit_records) - before_internal, '셀')

## 3. 공통 HTTP·공공 API 함수

표준 라이브러리만 사용해 재시도, 지수형 대기, 페이지네이션, CSV 캐시를 처리합니다. 인증키는 노트북에 저장하지 않습니다.

In [ ]:
USER_AGENT = 'Mozilla/5.0 (compatible; BondMissingnessResearch/1.0; academic-analysis)'

def build_data_go_kr_url(endpoint, params, service_key):
    params = {k: v for k, v in params.items() if v is not None and str(v) != '' and k != 'serviceKey'}
    key_fragment = service_key if '%' in service_key else quote_plus(service_key)
    query = urlencode(params, doseq=True)
    return endpoint + '?serviceKey=' + key_fragment + ('&' + query if query else '')

def http_get_bytes(url, timeout=45, retries=4):
    last_error = None
    for attempt in range(retries):
        try:
            request = Request(url, headers={'User-Agent': USER_AGENT, 'Accept': 'application/json,text/html,application/xml;q=0.9,*/*;q=0.8'})
            with urlopen(request, timeout=timeout) as response:
                return response.read()
        except (HTTPError, URLError, TimeoutError, ConnectionError) as exc:
            last_error = exc
            if attempt + 1 < retries:
                time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f'HTTP 요청 실패: {url[:160]} / {last_error}')

def json_from_bytes(raw):
    text = raw.decode('utf-8-sig', errors='replace').strip()
    if not text.startswith(('{', '[')):
        raise ValueError('JSON이 아닌 응답: ' + text[:500])
    return json.loads(text)

def extract_public_items(payload):
    root = payload.get('response', payload) if isinstance(payload, dict) else {}
    header = root.get('header', {}) if isinstance(root, dict) else {}
    code = str(header.get('resultCode', '00'))
    if code not in ('00', '0', '0000', ''):
        raise RuntimeError(f'공공 API 오류 {code}: {header.get("resultMsg")}')
    body = root.get('body', {}) if isinstance(root, dict) else {}
    items = body.get('items', []) if isinstance(body, dict) else []
    if isinstance(items, dict):
        items = items.get('item', items)
    if items is None:
        items = []
    if isinstance(items, dict):
        items = [items]
    return items, int(body.get('totalCount', len(items)) or 0)

def fetch_public_paginated(endpoint, base_params, cache_path, service_key, page_size=1000):
    cache_path = Path(cache_path)
    if cache_path.exists():
        print('캐시 사용:', cache_path.name)
        return pd.read_csv(cache_path, dtype=str)
    if not service_key:
        return pd.DataFrame()
    all_items = []
    page = 1
    total = None
    while total is None or len(all_items) < total:
        params = dict(base_params)
        params.update({'pageNo': page, 'numOfRows': page_size, 'resultType': 'json'})
        url = build_data_go_kr_url(endpoint, params, service_key)
        payload = json_from_bytes(http_get_bytes(url))
        items, total = extract_public_items(payload)
        if not items:
            break
        all_items.extend(items)
        print(f'  page {page}: {len(all_items):,}/{total:,}')
        page += 1
    result = pd.DataFrame(all_items)
    if WRITE_OUTPUTS and not result.empty:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        result.to_csv(cache_path, index=False, encoding='utf-8-sig')
    return result

def parse_number(value, percent=False):
    if not candidate_available(value):
        return np.nan
    text = normalize_text(value).replace(',', '')
    match = re.search(r'-?[0-9]+(?:\.[0-9]+)?', text)
    if not match:
        return np.nan
    number = float(match.group())
    return number

def parse_amount_won(value):
    if not candidate_available(value):
        return np.nan
    text = normalize_text(value).replace(',', '')
    number = parse_number(text)
    if pd.isna(number):
        return np.nan
    units = [('조', 10**12), ('억', 10**8), ('만', 10**4)]
    for token, multiplier in units:
        if token in text:
            return number * multiplier
    return number

## 4. SEIBro ISIN 상세정보 — 인증키 불필요

현재 결측이 남아 있는 정적 변수의 ISIN만 조회합니다. 각 ISIN의 원문 표를 JSON 캐시에 저장합니다. 발행일, 만기일, 표면금리, 발행금액·잔액, 발행자, 채권분류, 공모/사모, 통화, 이자방식을 보완할 수 있습니다. 영구채의 만기일은 데이터 정의상 1차 콜행사개시일이어야 하므로, SEIBro 법정만기와 의미가 다를 가능성이 있는 행은 자동 채우지 않고 검토 후보로 보냅니다.

In [ ]:
SEIBRO_URL = 'https://m.seibro.or.kr/cnts/bond/selectDetailSearch.do'
SEIBRO_TARGETS = ['pd_nm', 'bd_knd', 'curr_cd', 'pd_pbcm', 'isu_dt', 'mat_dt', 'srfc_irt', 'bd_tisu_a', 'isu_bal_amt', 'bd_ofr_tcd', 'bd_inrt_tcd', 'bd_intp_tcd']

def parse_seibro_html(html):
    pairs = {}
    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        tables = []
    for table in tables:
        if table.shape[1] < 2:
            continue
        for _, row in table.iterrows():
            label = normalize_text(row.iloc[0])
            value = normalize_text(row.iloc[1])
            if label and label not in pairs:
                pairs[label] = value
    return pairs

def get_seibro_detail(isin):
    cache_path = SEIBRO_CACHE_DIR / f'{isin}.json'
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding='utf-8'))
    url = SEIBRO_URL + '?' + urlencode({'InOrdSel': 'all', 'txt_code': isin, 'txt_sch': ''})
    raw = http_get_bytes(url, timeout=45, retries=4)
    html = raw.decode('utf-8', errors='replace')
    pairs = parse_seibro_html(html)
    if WRITE_OUTPUTS:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(json.dumps(pairs, ensure_ascii=False, indent=2), encoding='utf-8')
    return pairs

def seibro_candidates(pairs):
    result = {
        'pd_nm': pairs.get('종목명'),
        'bd_knd': pairs.get('채권분류'),
        'curr_cd': pairs.get('발행통화'),
        'pd_pbcm': pairs.get('발행자'),
        'isu_dt': normalize_date(pairs.get('발행일')),
        'mat_dt': normalize_date(pairs.get('만기일')),
        'srfc_irt': parse_number(pairs.get('표면금리')),
        'bd_tisu_a': parse_amount_won(pairs.get('발행금액')),
        'isu_bal_amt': parse_amount_won(pairs.get('발행잔액')),
        'bd_ofr_tcd': pairs.get('공모/사모')
    }
    interest = normalize_text(pairs.get('이자지급방식'))
    if '-' in interest:
        left, right = interest.split('-', 1)
        result['bd_inrt_tcd'] = left
        result['bd_intp_tcd'] = right
    elif interest:
        result['bd_intp_tcd'] = interest
    return result

before_seibro = len(audit_records)
seibro_errors = []
if RUN_SEIBRO and 'pd_no' in df.columns:
    missing_static = pd.Series(False, index=df.index)
    for column in [c for c in SEIBRO_TARGETS if c in df.columns]:
        missing_static |= df[column].isna()
    needed_isins = df.loc[missing_static, 'pd_no'].dropna().astype(str).drop_duplicates().tolist()[:SEIBRO_MAX_ISINS]
    print('SEIBro 조회 대상 ISIN:', len(needed_isins))
    for n, isin in enumerate(needed_isins, start=1):
        try:
            pairs = get_seibro_detail(isin)
            if normalize_text(pairs.get('종목코드')) != isin:
                seibro_errors.append({'pd_no': isin, 'error': '종목코드 불일치 또는 결과 없음'})
                continue
            candidates = seibro_candidates(pairs)
            target_rows = df.index[df['pd_no'].astype(str).eq(isin)]
            is_perpetual = any(token in normalize_text(pairs.get('채권분류')) + normalize_text(pairs.get('종목명')) for token in ['영구', '신종자본'])
            for idx in target_rows:
                for column, value in candidates.items():
                    if column == 'mat_dt' and is_perpetual and pd.isna(df.at[idx, column]):
                        record_review(idx, column, value, 'SEIBro', datetime.now().date().isoformat(), f'pd_no={isin}', 'REVIEW', '영구채는 데이터 정의상 1차 콜행사개시일 필요')
                        continue
                    safe_fill(idx, column, value, 'SEIBro', datetime.now().date().isoformat(), f'pd_no={isin}', 'EXACT_ISIN', 'SEIBro 종목코드가 pd_no와 정확히 일치')
        except Exception as exc:
            seibro_errors.append({'pd_no': isin, 'error': str(exc)})
        if n % 25 == 0:
            print(f'  {n}/{len(needed_isins)} 처리')
        if n < len(needed_isins):
            time.sleep(SEIBRO_DELAY_SECONDS)
print('SEIBro 자동 보완:', len(audit_records) - before_seibro, '셀 / 오류:', len(seibro_errors))
if seibro_errors:
    display(pd.DataFrame(seibro_errors).head(50))

## 5. 금융위원회·한국예탁결제원 채권기본정보 API

기준일 전체 목록을 먼저 받고, 결측 ISIN만 정확히 조인합니다. API가 기준일 전체 목록을 제공하지 않는 경우 결측이 남은 ISIN을 개별 조회합니다. 공개 API와 데이터셋의 분류체계가 다른 값은 자동으로 억지 변환하지 않습니다.

In [ ]:
BOND_BASIC_ENDPOINT = 'https://apis.data.go.kr/1160100/GetBondIssuInfoService_V2/getBondBasiInfo_V2'
BASIC_SOURCE_MAP = {
    'isinCdNm': 'pd_nm', 'bondIsurNm': 'pd_pbcm',
    'bondIssuDt': 'isu_dt', 'bondExprDt': 'mat_dt',
    'bondSrfcInrt': 'srfc_irt', 'bondIssuAmt': 'bd_tisu_a',
    'bondBal': 'isu_bal_amt', 'bondOffrMcdNm': 'bd_ofr_tcd',
    'bondIssuCurCd': 'curr_cd', 'pclrBondKcdNm': 'bd_knd'
}

def clean_basic_api_value(source_column, value):
    if source_column in ('bondIssuDt', 'bondExprDt'):
        return normalize_date(value)
    if source_column in ('bondSrfcInrt', 'bondIssuAmt', 'bondBal'):
        return parse_number(value)
    return value

def fetch_basic_by_isin(isin):
    cache = CACHE_DIR / 'bond_basic_by_isin' / f'{isin}.csv'
    return fetch_public_paginated(BOND_BASIC_ENDPOINT, {'isinCd': isin}, cache, BOND_BASIC_SERVICE_KEY, page_size=100)

before_basic_api = len(audit_records)
basic_api = pd.DataFrame()
if RUN_PUBLIC_DATA_API and BOND_BASIC_SERVICE_KEY:
    snapshot_values = [date8(v) for v in df_original['info_base_dt'].dropna()] if 'info_base_dt' in df_original else []
    snapshot_date = max(snapshot_values) if snapshot_values else datetime.now().strftime('%Y%m%d')
    basic_api = fetch_public_paginated(BOND_BASIC_ENDPOINT, {'basDt': snapshot_date}, CACHE_DIR / f'bond_basic_{snapshot_date}.csv', BOND_BASIC_SERVICE_KEY, PUBLIC_API_PAGE_SIZE)
    required_targets = [c for c in BASIC_SOURCE_MAP.values() if c in df.columns]
    unresolved_mask = df[required_targets].isna().any(axis=1) if required_targets else pd.Series(False, index=df.index)
    unresolved_isins = df.loc[unresolved_mask, 'pd_no'].dropna().astype(str).drop_duplicates().tolist()[:PUBLIC_API_MAX_ISINS]
    present = set(basic_api.get('isinCd', pd.Series(dtype=str)).dropna().astype(str)) if not basic_api.empty else set()
    missing_from_bulk = [isin for isin in unresolved_isins if isin not in present]
    if missing_from_bulk:
        extra = []
        print('기준일 목록에 없는 결측 ISIN 개별 조회:', len(missing_from_bulk))
        for n, isin in enumerate(missing_from_bulk, 1):
            try:
                part = fetch_basic_by_isin(isin)
                if not part.empty:
                    extra.append(part)
            except Exception as exc:
                print('  실패', isin, str(exc)[:120])
            if n % 100 == 0:
                print(f'  {n}/{len(missing_from_bulk)}')
        if extra:
            basic_api = pd.concat([basic_api] + extra, ignore_index=True).drop_duplicates()
    if not basic_api.empty and 'isinCd' in basic_api.columns:
        basic_api['isinCd'] = basic_api['isinCd'].astype(str)
        for isin, group in basic_api.groupby('isinCd'):
            target_rows = df.index[df['pd_no'].astype(str).eq(isin)]
            if len(target_rows) == 0:
                continue
            for source_col, target_col in BASIC_SOURCE_MAP.items():
                if source_col not in group or target_col not in df:
                    continue
                values = [clean_basic_api_value(source_col, v) for v in group[source_col] if candidate_available(v)]
                value = canonical_unique(values)
                if value is None:
                    for idx in target_rows[df.loc[target_rows, target_col].isna()]:
                        if len(values) > 1:
                            record_review(idx, target_col, '|'.join(map(str, values[:10])), 'data.go.kr_KSD_bond_basic', snapshot_date, f'pd_no={isin}', 'CONFLICT', '같은 ISIN에 후보가 여러 개')
                    continue
                for idx in target_rows:
                    safe_fill(idx, target_col, value, 'data.go.kr_KSD_bond_basic', snapshot_date, f'isinCd={isin}', 'EXACT_ISIN', f'원천 필드 {source_col}')
else:
    print('공공데이터포털 인증키가 없어 채권기본정보 API를 생략했습니다.')
print('채권기본정보 API 자동 보완:', len(audit_records) - before_basic_api, '셀')

## 6. 금융위원회·한국거래소 채권시세 API

장내 행만 대상으로 동일 ISIN·동일 기준일 시세를 연결합니다. 거래량이 0이거나 종가가 0인 날은 값을 만들지 않습니다. 동일 ISIN이 여러 KRX 시장에 존재하면서 시세가 충돌하면 검토 후보로 보냅니다. 장외 행의 장내 종가 결측은 구조적 비해당으로 유지됩니다.

In [ ]:
KRX_PRICE_ENDPOINT = 'https://apis.data.go.kr/1160100/service/GetBondSecuritiesInfoService/getBondPriceInfo'
before_krx = len(audit_records)
krx_price = pd.DataFrame()
if RUN_KRX_PRICE_API and KRX_BOND_PRICE_SERVICE_KEY and {'pd_exg_mkt', 'pd_no', 'info_base_dt'}.issubset(df.columns):
    eligible = df['pd_exg_mkt'].astype(str).eq('장내')
    eligible &= df[[c for c in ['exg_close_price', 'exg_close_yield', 'exg_close_price_base_dt'] if c in df]].isna().any(axis=1)
    for base_dt in sorted({date8(v) for v in df.loc[eligible, 'info_base_dt'] if date8(v)}):
        part = fetch_public_paginated(KRX_PRICE_ENDPOINT, {'basDt': base_dt}, CACHE_DIR / f'krx_bond_price_{base_dt}.csv', KRX_BOND_PRICE_SERVICE_KEY, PUBLIC_API_PAGE_SIZE)
        if not part.empty:
            krx_price = pd.concat([krx_price, part], ignore_index=True)
    if not krx_price.empty:
        krx_price['isinCd'] = krx_price['isinCd'].astype(str)
        krx_price['basDt'] = krx_price['basDt'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(8)
        for (isin, base_dt), group in krx_price.groupby(['isinCd', 'basDt']):
            rows = df.index[eligible & df['pd_no'].astype(str).eq(isin) & df['info_base_dt'].map(date8).eq(base_dt)]
            if len(rows) == 0:
                continue
            group = group.copy()
            group['_trqu'] = pd.to_numeric(group.get('trqu'), errors='coerce')
            group['_price'] = pd.to_numeric(group.get('clprPrc'), errors='coerce')
            group['_yield'] = pd.to_numeric(group.get('clprBnfRt'), errors='coerce')
            traded = group[(group['_trqu'] > 0) & (group['_price'] > 0)]
            price_values = traded['_price'].dropna().unique()
            yield_values = traded['_yield'].dropna().unique()
            for idx in rows:
                if len(price_values) == 1:
                    safe_fill(idx, 'exg_close_price', price_values[0], 'data.go.kr_KRX_bond_price', base_dt, f'isinCd={isin}|basDt={base_dt}', 'EXACT_ISIN_DATE_TRADED')
                    safe_fill(idx, 'exg_close_price_base_dt', normalize_date(base_dt), 'data.go.kr_KRX_bond_price', base_dt, f'isinCd={isin}|basDt={base_dt}', 'EXACT_ISIN_DATE_TRADED')
                elif len(price_values) > 1:
                    record_review(idx, 'exg_close_price', '|'.join(map(str, price_values)), 'data.go.kr_KRX_bond_price', base_dt, f'isinCd={isin}', 'MARKET_CONFLICT', 'KRX 시장별 종가 충돌')
                if len(yield_values) == 1:
                    safe_fill(idx, 'exg_close_yield', yield_values[0], 'data.go.kr_KRX_bond_price', base_dt, f'isinCd={isin}|basDt={base_dt}', 'EXACT_ISIN_DATE_TRADED')
                elif len(yield_values) > 1:
                    record_review(idx, 'exg_close_yield', '|'.join(map(str, yield_values)), 'data.go.kr_KRX_bond_price', base_dt, f'isinCd={isin}', 'MARKET_CONFLICT', 'KRX 시장별 종가수익률 충돌')
else:
    print('KRX 시세 API: 인증키 없음, 실행 해제, 또는 대상 없음')
print('KRX 시세 API 자동 보완:', len(audit_records) - before_krx, '셀')

## 7. 선택 외부 파일 연결

정확한 상용·내부 데이터가 있을 때 가장 많은 결측을 채우는 단계입니다. 파일은 CSV 또는 XLSX를 지원합니다.

- RATINGS_FILE: pd_no, info_base_dt, crd_grd, crd_grd_dt. info_base_dt 현재 유효한 종목등급으로 정리된 스냅샷만 사용하십시오. 발행사등급을 종목등급으로 대신 쓰면 안 됩니다.
- PRICING_FILE: pd_no, info_base_dt와 평가 변수들. 반드시 같은 평가사·같은 기준일·같은 현금흐름 규칙이어야 합니다.
- INTERNAL_SALES_FILE: pd_no, info_base_dt, pd_exg_mkt, info_seq와 13개 판매 변수. 원본 LOT를 정확히 식별해야 합니다.

키가 중복되면서 값이 충돌하면 자동으로 채우지 않고 검토 후보에 기록합니다.

In [ ]:
SALES_COLUMNS = [
    'after_tax_yield', 'avg_annual_tax_yield', 'bdbns_abl_chnl_nm',
    'bdbns_abl_chnl_tcd', 'buyable_quantity', 'buy_yield',
    'corp_after_tax_yield', 'corp_pretax_yield', 'depo_equiv_yield_154',
    'depo_equiv_yield_495', 'pref_tax_yield', 'sale_yield_base_dt', 'trade_price'
]
PRICING_COLUMNS = ['applied_yield', 'cov', 'dirty', 'dur', 'eval_price', 'ndy_applied_yield', 'ndy_cov', 'ndy_dirty', 'ndy_dur', 'ndy_eval_price']

def read_table(path):
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    return pd.read_csv(path)

def normalized_key_frame(frame, keys):
    result = frame.copy()
    for key in keys:
        if key not in result.columns:
            continue
        if key.endswith('_dt'):
            result[key] = result[key].map(date8)
        else:
            result[key] = result[key].map(normalize_text)
    return result

def apply_exact_file(path, keys, target_columns, source_name):
    if not path:
        print(source_name, ': 파일 미지정 - 생략')
        return 0
    ext = read_table(path)
    missing_keys = [k for k in keys if k not in ext.columns or k not in df.columns]
    if missing_keys:
        raise ValueError(f'{source_name} 필수 키 누락: {missing_keys}')
    targets = [c for c in target_columns if c in ext.columns and c in df.columns]
    if not targets:
        print(source_name, ': 채울 대상 열 없음')
        return 0
    left = normalized_key_frame(df.reset_index(names='_row_index'), keys)
    right = normalized_key_frame(ext, keys)
    before = len(audit_records)
    grouped = right.groupby(keys, dropna=False)
    for key_values, group in grouped:
        key_values = key_values if isinstance(key_values, tuple) else (key_values,)
        mask = pd.Series(True, index=left.index)
        for key, value in zip(keys, key_values):
            mask &= left[key].eq(value)
        rows = left.loc[mask, '_row_index'].astype(int).tolist()
        if not rows:
            continue
        match_key = '|'.join(f'{k}={v}' for k, v in zip(keys, key_values))
        for column in targets:
            values = [v for v in group[column] if candidate_available(v)]
            value = canonical_unique(values)
            for idx in rows:
                if value is not None:
                    safe_fill(idx, column, value, source_name, date8(df_original.at[idx, 'info_base_dt']) if 'info_base_dt' in df_original else '', match_key, 'EXACT_FILE_KEY')
                elif len(values) > 1 and pd.isna(df.at[idx, column]):
                    record_review(idx, column, '|'.join(map(str, values[:20])), source_name, '', match_key, 'FILE_CONFLICT', '동일 키에 값이 여러 개')
    filled = len(audit_records) - before
    print(source_name, ':', filled, '셀 자동 보완')
    return filled

apply_exact_file(RATINGS_FILE, ['pd_no', 'info_base_dt'], ['crd_grd', 'crd_grd_dt'], 'external_issue_rating_file')
apply_exact_file(PRICING_FILE, ['pd_no', 'info_base_dt'], PRICING_COLUMNS, 'external_pricing_vendor_file')
sales_keys = [k for k in ['pd_no', 'info_base_dt', 'pd_exg_mkt', 'info_seq'] if k in df.columns]
apply_exact_file(INTERNAL_SALES_FILE, sales_keys, SALES_COLUMNS, 'internal_sales_inventory_file')

## 8. 안전한 파생값

원천값이 확정된 경우에만 결정적으로 계산 가능한 값을 보완합니다. 가격, 수익률, 듀레이션, 컨벡시티는 평가 규칙이 필요하므로 임의 계산하지 않습니다.

In [ ]:
before_derived = len(audit_records)
if {'pd_no', 'pd_ctry_cd'}.issubset(df.columns):
    for idx in df.index[df['pd_ctry_cd'].isna()]:
        isin = normalize_text(df.at[idx, 'pd_no'])
        if re.fullmatch(r'[A-Z]{2}[A-Z0-9]{10}', isin):
            safe_fill(idx, 'pd_ctry_cd', isin[:2], 'derived_from_ISIN_ISO6166', '', f'pd_no={isin}', 'DETERMINISTIC', 'ISIN 앞 2자리 국가코드')

if {'remaining_days', 'mat_dt', 'info_base_dt'}.issubset(df.columns):
    for idx in df.index[df['remaining_days'].isna()]:
        maturity = normalize_date(df.at[idx, 'mat_dt'])
        base = normalize_date(df.at[idx, 'info_base_dt'])
        if maturity is not None and base is not None:
            days = int((maturity - base).days)
            if days >= 0:
                safe_fill(idx, 'remaining_days', days, 'derived_from_mat_dt_info_base_dt', date8(base), f'row_key={ROW_KEYS.at[idx]}', 'DETERMINISTIC', '상환일자 - 기준일자')
print('결정적 파생값 보완:', len(audit_records) - before_derived, '셀')

## 9. 남은 결측의 의미 분류

남은 NULL을 단순 실패로 보지 않고 구조적 비해당, 내부원장 필요, 평가사 필요, 신용등급 검토 필요, 공개정보 미확보로 나눕니다. 결과는 셀 단위로 저장됩니다.

In [ ]:
EVALUATION_COLUMNS = ['cov', 'dirty', 'dur', 'eval_price', 'ndy_applied_yield', 'ndy_cov', 'ndy_dirty', 'ndy_dur', 'ndy_eval_price']
EXCHANGE_COLUMNS = ['exg_close_price', 'exg_close_price_base_dt', 'exg_close_yield']
NAME_COLUMNS = ['pd_eng_nm', 'pd_abrv_nm', 'pd_abrv_eng_nm']
SOVEREIGN_TOKENS = ['국채', '지방채', '통안', '한국은행', '국고', '국민주택']

def classify_unresolved(idx, column):
    market = normalize_text(df.at[idx, 'pd_exg_mkt']) if 'pd_exg_mkt' in df else ''
    kind = normalize_text(df.at[idx, 'bd_knd']) if 'bd_knd' in df else ''
    name = normalize_text(df.at[idx, 'pd_nm']) if 'pd_nm' in df else ''
    if column in SALES_COLUMNS:
        return 'INTERNAL_SALES_INVENTORY_REQUIRED', '판매·재고 LOT 원장만 정확한 원천'
    if column in EXCHANGE_COLUMNS and market != '장내':
        return 'STRUCTURAL_NOT_APPLICABLE', '장외 행에는 장내 종가가 적용되지 않음'
    if column in EXCHANGE_COLUMNS:
        return 'KRX_TRADE_NOT_FOUND', '동일 ISIN·동일일자 유효 체결 시세 미확보'
    if column in EVALUATION_COLUMNS or column == 'applied_yield':
        return 'PRICING_VENDOR_REQUIRED', '동일 평가사·기준일·평가규칙 데이터 필요'
    if column in ('crd_grd', 'crd_grd_dt') and any(token in kind + name for token in SOVEREIGN_TOKENS):
        return 'STRUCTURAL_NOT_RATED_OR_NOT_REQUIRED', '국공채·중앙은행채 등 무등급 가능성'
    if column in ('crd_grd', 'crd_grd_dt'):
        return 'ISSUE_RATING_SOURCE_REQUIRED', '종목별 유효 신용등급 원천 필요'
    if column in NAME_COLUMNS:
        return 'PROVIDER_NAMING_UNAVAILABLE', '외부 명칭은 내부 제공자 약어와 다를 수 있음'
    return 'PUBLIC_SOURCE_NOT_FOUND', '선택한 공개 원천에서 정확한 값 미확보'

unresolved_records = []
id_columns = [c for c in ['pd_no', 'pd_nm', 'info_base_dt', 'pd_exg_mkt', 'info_seq', 'bd_knd'] if c in df.columns]
for column in df.columns[df.isna().any()]:
    for idx in df.index[df[column].isna()]:
        status, reason = classify_unresolved(idx, column)
        record = {'row_index': int(idx), 'row_key': ROW_KEYS.at[idx], 'column': column, 'status': status, 'reason': reason}
        record.update({c: df.at[idx, c] for c in id_columns})
        unresolved_records.append(record)
unresolved_df = pd.DataFrame(unresolved_records)
display(unresolved_df.groupby(['column', 'status']).size().rename('count').reset_index().sort_values('count', ascending=False))

## 10. 품질검사와 결과 저장

다음 검사가 하나라도 실패하면 저장을 중단합니다: 행·열·순서 보존, 원본 비결측값 불변, 원본 0 불변, 새 NULL 미생성, 감사 로그와 실제 보완 셀 수 일치.

In [ ]:
assert len(df) == len(df_original), '행 수가 변했습니다.'
assert df.columns.tolist() == original_columns, '열 또는 열 순서가 변했습니다.'

protected_change_count = 0
for column in original_columns:
    mask = original_nonnull[column]
    left = df_original.loc[mask, column].astype(str).fillna('<NA>')
    right = df.loc[mask, column].astype(str).fillna('<NA>')
    protected_change_count += int((left != right).sum())
assert protected_change_count == 0, f'원본 비결측값 {protected_change_count}개가 변경되었습니다.'

original_zero_change_count = 0
for column in original_columns:
    numeric_original = pd.to_numeric(df_original[column], errors='coerce')
    zero_mask = numeric_original.eq(0) & df_original[column].notna()
    if zero_mask.any():
        original_zero_change_count += int((df.loc[zero_mask, column].astype(str) != df_original.loc[zero_mask, column].astype(str)).sum())
assert original_zero_change_count == 0, f'원본 0 값 {original_zero_change_count}개가 변경되었습니다.'

new_null_count = int((df.isna() & original_nonnull).sum().sum())
assert new_null_count == 0, f'새 NULL {new_null_count}개가 생겼습니다.'

before_missing = df_original.isna().sum()
after_missing = df.isna().sum()
filled_by_column = before_missing - after_missing
actual_filled = int(filled_by_column.sum())
audit_df = pd.DataFrame(audit_records)
review_df = pd.DataFrame(review_records)
assert actual_filled == len(audit_df), f'실제 보완 {actual_filled}개와 감사 로그 {len(audit_df)}개가 다릅니다.'

summary_df = pd.DataFrame({
    'column': original_columns,
    'before_null': [int(before_missing[c]) for c in original_columns],
    'filled': [int(filled_by_column[c]) for c in original_columns],
    'after_null': [int(after_missing[c]) for c in original_columns],
    'after_null_pct': [round(100 * after_missing[c] / len(df), 4) for c in original_columns]
}).sort_values(['filled', 'after_null'], ascending=False)

source_summary = (audit_df.groupby(['source', 'column']).size().rename('filled_cells').reset_index()
                  if not audit_df.empty else pd.DataFrame(columns=['source', 'column', 'filled_cells']))
qc_df = pd.DataFrame([
    {'test': 'row_count_unchanged', 'value': len(df), 'passed': len(df) == len(df_original)},
    {'test': 'column_order_unchanged', 'value': len(df.columns), 'passed': df.columns.tolist() == original_columns},
    {'test': 'protected_original_changes', 'value': protected_change_count, 'passed': protected_change_count == 0},
    {'test': 'original_zero_changes', 'value': original_zero_change_count, 'passed': original_zero_change_count == 0},
    {'test': 'new_nulls', 'value': new_null_count, 'passed': new_null_count == 0},
    {'test': 'filled_cells', 'value': actual_filled, 'passed': actual_filled == len(audit_df)},
])

print(f'자동 보완 합계: {actual_filled:,}셀')
print(f'남은 NULL: {int(df.isna().sum().sum()):,}셀')
display(qc_df)
display(summary_df[summary_df['before_null'] > 0])
display(source_summary)

if WRITE_OUTPUTS:
    filled_path = OUTPUT_DIR / '국내채권마스터_filled.xlsx'
    report_path = OUTPUT_DIR / 'prbd01n001_fill_report.xlsx'
    df.to_excel(filled_path, index=False)
    with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='column_summary', index=False)
        source_summary.to_excel(writer, sheet_name='source_summary', index=False)
        qc_df.to_excel(writer, sheet_name='quality_checks', index=False)
        audit_df.to_excel(writer, sheet_name='fill_audit', index=False)
        review_df.to_excel(writer, sheet_name='review_candidates', index=False)
    audit_df.to_csv(OUTPUT_DIR / 'prbd01n001_fill_audit.csv', index=False, encoding='utf-8-sig')
    review_df.to_csv(OUTPUT_DIR / 'prbd01n001_review_candidates.csv', index=False, encoding='utf-8-sig')
    unresolved_df.to_csv(OUTPUT_DIR / 'prbd01n001_unresolved_cells.csv', index=False, encoding='utf-8-sig')
    if seibro_errors:
        pd.DataFrame(seibro_errors).to_csv(OUTPUT_DIR / 'seibro_errors.csv', index=False, encoding='utf-8-sig')
    print('저장 완료:')
    print(' ', filled_path)
    print(' ', report_path)
    print(' ', OUTPUT_DIR / 'prbd01n001_unresolved_cells.csv')
else:
    print('WRITE_OUTPUTS=0이므로 파일 저장을 생략했습니다.')

## 결과를 읽는 법

- 국내채권마스터.xlsx: 원본 열과 행을 그대로 유지한 보완 데이터 복사본
- fill_report.xlsx: 열별 보완 수, 출처별 보완 수, 품질검사, 셀 단위 감사 로그, 수동 검토 후보
- unresolved_cells.csv: 남은 모든 NULL의 행·열·원인 분류
- cache: API와 SEIBro 중간 결과. 재실행 시 호출량과 시간을 줄임

실제 분석에서는 원본과 filled_proposal을 모두 보관하고, 감사 로그가 있는 셀만 외부 보완값으로 간주하십시오. STRUCTURAL_NOT_APPLICABLE 또는 STRUCTURAL_NOT_RATED_OR_NOT_REQUIRED는 결측을 억지로 채우지 않는 것이 올바른 처리입니다.